# scMulan — 基础模型教程

**scMulan** — 原生多组学联合建模（同时支持 RNA+ATAC+蛋白质），专为 CITE-seq/10x Multiome 设计

| 属性 | 值 |
|----------|-------|
| **任务** | embed, integrate |
| **物种** | human |
| **基因 ID** | symbol |
| **需要 GPU** | 是 |
| **最低显存** | 16 GB |
| **嵌入维度** | 512 |
| **代码仓库** | [https://github.com/SuperBianC/scMulan](https://github.com/SuperBianC/scMulan) |


> **注意：** scMulan 原生支持多组学数据（RNA+ATAC+蛋白质）。对于单模态 RNA 数据，其他模型可能更为合适。

本教程演示如何通过统一的 `ov.fm` API 使用 **scMulan**。

**引用：** Zeng, Z. et al. (2024). OmicVerse: a framework for bridging and deepening insights across bulk and single-cell sequencing. *Nature Communications*, 15(1), 5983.

In [ ]:
import omicverse as ov
import scanpy as sc
import os
import warnings
warnings.filterwarnings('ignore')

ov.plot_set()

## 多组学最佳实践

在使用 scMulan 处理多组学数据时：

1. **CITE-seq** — RNA + 表面蛋白：将蛋白质计数存储于 `adata.obsm['protein_expression']`
2. **10x Multiome** — RNA + ATAC：使用 MuData，通过 `mdata.mod['rna']` 和 `mdata.mod['atac']` 访问
3. **联合嵌入** — scMulan 在所有模态间创建统一的嵌入空间
4. **批次整合** — 多样本实验请传入 `batch_key`

## 步骤 1：查看模型规格

使用 `ov.fm.describe_model()` 获取 scMulan 的完整规格信息。

In [ ]:
info = ov.fm.describe_model("scmulan")

print("=== Model Info ===")
print(f"Name: {info['model']['name']}")
print(f"Version: {info['model']['version']}")
print(f"Tasks: {info['model']['tasks']}")
print(f"Species: {info['model']['species']}")
print(f"Embedding dim: {info['model']['embedding_dim']}")
print(f"Differentiator: {info['model']['differentiator']}")

print("\n=== Input Contract ===")
print(f"Gene ID scheme: {info['input_contract']['gene_id_scheme']}")
print(f"Preprocessing: {info['input_contract']['preprocessing']}")

print("\n=== Output Contract ===")
print(f"Embedding key: {info['output_contract']['embedding_key']}")
print(f"Embedding dim: {info['output_contract']['embedding_dim']}")

## 步骤 2：准备数据

加载数据集并将其保存，以供 `ov.fm` 工作流使用。大多数基础模型需要原始计数（非负值）。

In [ ]:
# scMulan is designed for multi-omics (CITE-seq, 10x Multiome).
# For RNA-only data, it still works but other models may be preferred.
#
# For multi-omics data, organize as MuData:
# import mudata as mu
# mdata = mu.read('multiome_data.h5mu')

adata = sc.datasets.pbmc3k()
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
print(f'Dataset: {adata.n_obs} cells x {adata.n_vars} genes')

adata.write_h5ad('pbmc3k_scmulan.h5ad')


## 步骤 3：分析数据并验证兼容性

在运行推理之前，检查您的数据是否与 scMulan 兼容。

In [ ]:
profile = ov.fm.profile_data("pbmc3k_scmulan.h5ad")

print("=== Data Profile ===")
print(f"Species: {profile['species']}")
print(f"Gene scheme: {profile['gene_scheme']}")
print(f"Modality: {profile['modality']}")
print(f"Cells: {profile['n_cells']:,}")
print(f"Genes: {profile['n_genes']:,}")

# Validate compatibility
validation = ov.fm.preprocess_validate("pbmc3k_scmulan.h5ad", "scmulan", "embed")
print(f"\n=== Validation: {validation['status']} ===")
for d in validation.get("diagnostics", []):
    print(f"  [{d['severity']}] {d['message']}")
if validation.get("auto_fixes"):
    print("\nSuggested fixes:")
    for fix in validation["auto_fixes"]:
        print(f"  - {fix}")

## 步骤 4：运行 scMulan 推理

通过 `ov.fm.run()` 执行 scMulan。该函数负责处理预处理、模型加载、推理和输出写入。

In [ ]:
result = ov.fm.run(
    task="embed",
    model_name="scmulan",
    adata_path="pbmc3k_scmulan.h5ad",
    output_path="pbmc3k_scmulan_out.h5ad",
    device="auto",
)

if "error" in result:
    print(f"Error: {result['error']}")
    if "suggestion" in result:
        print(f"Suggestion: {result['suggestion']}")
else:
    print(f"Status: {result['status']}")
    print(f"Output keys: {result.get('output_keys', [])}")
    print(f"Cells processed: {result.get('n_cells', 0)}")

## 步骤 5：可视化与结果解读

加载输出，从 scMulan 嵌入计算 UMAP，并评估质量。

In [ ]:
if os.path.exists("pbmc3k_scmulan_out.h5ad"):
    adata_out = sc.read_h5ad("pbmc3k_scmulan_out.h5ad")
    emb_key = "X_scmulan"
    
    if emb_key in adata_out.obsm:
        print(f"Embedding shape: {adata_out.obsm[emb_key].shape}")
        
        # UMAP visualization
        sc.pp.neighbors(adata_out, use_rep=emb_key)
        sc.tl.umap(adata_out)
        sc.tl.leiden(adata_out, resolution=0.5)
        sc.pl.umap(adata_out, color=["leiden"],
                   title="scMulan Embedding (PBMC 3k)")
        
        # QA metrics
        interpretation = ov.fm.interpret_results("pbmc3k_scmulan_out.h5ad", task="embed")
        if "embeddings" in interpretation["metrics"]:
            for k, v in interpretation["metrics"]["embeddings"].items():
                print(f"\n{k}: dim={v['dim']}", end="")
                if "silhouette" in v:
                    print(f", silhouette={v['silhouette']:.4f}", end="")
                print()
    else:
        print(f"Embedding key {emb_key} not found.")
        print(f"Available keys: {list(adata_out.obsm.keys())}")
else:
    print("Output file not found — check model installation and adapter status.")
    print("See the Guide page for installation instructions.")

## 总结

| 步骤 | 函数 | 功能说明 |
|------|----------|-------------|
| 1 | `ov.fm.describe_model("scmulan")` | 查看模型规格及输入/输出契约 |
| 2 | `sc.datasets.pbmc3k()` | 准备输入数据 |
| 3 | `ov.fm.profile_data()` + `preprocess_validate()` | 检查兼容性 |
| 4 | `ov.fm.run()` | 执行 scMulan 推理 |
| 5 | `ov.fm.interpret_results()` | 评估嵌入质量 |

完整的模型目录请参见 `ov.fm.list_models()` 或 [ov.fm API 概览](t_fm_guide.md)。
scMulan 的详细规格说明，请参见 [scMulan 指南](t_fm_scmulan_guide.md)。